# E-commerce Order Support Agent — Python 3.11.9 Compatible

This version is designed for your current environment.

## Important compatibility change

We **do not use**:

```python
from langchain.tools import tool
from langchain.agents import create_agent
```

because `create_agent` imports LangGraph internals, and your environment currently has a LangGraph/checkpoint compatibility conflict.

Instead we use:

```python
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
```

and create a small tool-calling loop using:

```python
llm.bind_tools(...)
```

This is still a genuine **tool-using AI agent**, and the LLM connection is still entirely through `langchain_openai`.

## Step 1 — Check Python and package versions

This notebook targets **Python 3.11.9**.

No LangGraph or `create_agent` import is required.

In [ ]:
import sys
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.version)

for pkg in [
    "langchain-core",
    "langchain-openai",
    "openai",
    "pandas",
    "python-dotenv",
]:
    try:
        print(f"{pkg}: {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg}: NOT INSTALLED")

## Step 2 — Imports and `.env`

Your `.env` file should contain:

```text
OPENAI_API_KEY=your_openai_api_key
```

In [ ]:
import os
import pandas as pd

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY was not found in the .env file.")

print("Imports loaded successfully.")

## Step 3 — Load the e-commerce dataset

Keep `agent_ecommerce_orders.csv` in the same folder as this notebook.

In [ ]:
df = pd.read_csv("agent_ecommerce_orders.csv")

display(df.head())
print("Rows:", len(df))

## Step 4 — Create tools

A tool is a Python function the model is allowed to call.

We create four tools:

- Order status
- Order amount
- Return eligibility
- Issue details

In [ ]:
@tool
def get_order_status(order_id: str) -> str:
    """Get the current status and expected delivery date for an order ID."""
    order_id = str(order_id).strip().upper()
    row = df[df["order_id"].str.upper() == order_id]

    if row.empty:
        return f"Order {order_id} was not found."

    r = row.iloc[0]
    return (
        f"Order {r['order_id']} for {r['product']} has status '{r['status']}'. "
        f"Expected delivery: {r['expected_delivery']}."
    )


@tool
def get_order_amount(order_id: str) -> str:
    """Get the purchase amount and payment method for an order ID."""
    order_id = str(order_id).strip().upper()
    row = df[df["order_id"].str.upper() == order_id]

    if row.empty:
        return f"Order {order_id} was not found."

    r = row.iloc[0]
    return (
        f"Order {r['order_id']} amount is INR {r['amount_inr']} "
        f"paid using {r['payment_method']}."
    )


@tool
def check_return_eligibility(order_id: str) -> str:
    """Check return eligibility using the demo 10-day return policy."""
    order_id = str(order_id).strip().upper()
    row = df[df["order_id"].str.upper() == order_id]

    if row.empty:
        return f"Order {order_id} was not found."

    r = row.iloc[0]

    if r["status"] != "Delivered":
        return (
            f"Order {r['order_id']} is not delivered, "
            "so return eligibility cannot yet be applied."
        )

    delivery_date = pd.to_datetime(r["expected_delivery"], errors="coerce")
    reference_date = pd.Timestamp("2026-09-10")

    if pd.isna(delivery_date):
        return "Delivery date is unavailable."

    days = (reference_date - delivery_date).days

    if days <= 10:
        return f"Eligible for return. Delivered {days} day(s) ago."

    return f"Not eligible under the 10-day return policy. Delivered {days} day(s) ago."


@tool
def get_issue_details(order_id: str) -> str:
    """Check whether an issue is recorded for an order ID."""
    order_id = str(order_id).strip().upper()
    row = df[df["order_id"].str.upper() == order_id]

    if row.empty:
        return f"Order {order_id} was not found."

    r = row.iloc[0]

    if str(r["issue_reported"]).lower() == "yes":
        return f"Issue reported for {r['order_id']}: {r['issue_details']}."

    return f"No issue is currently recorded for {r['order_id']}."

## Step 5 — Create the model and bind the tools

`bind_tools()` tells the OpenAI model which Python tools are available.

The model can then decide:

- Whether a tool is necessary.
- Which tool should be called.
- What arguments should be passed.

In [ ]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
)

tools = [
    get_order_status,
    get_order_amount,
    check_return_eligibility,
    get_issue_details,
]

tool_map = {t.name: t for t in tools}

llm_with_tools = llm.bind_tools(tools)

print("Tools:", list(tool_map))

## Step 6 — Create the agent loop

The logic is:

```text
User
  ↓
LLM
  ↓
Tool call required?
  ├─ No  → final response
  └─ Yes → execute tool
             ↓
          return result to LLM
             ↓
          LLM decides again
```

This is the basic agent loop without `create_agent`.

In [ ]:
SYSTEM_PROMPT = """
You are an ecommerce customer-support agent.

Rules:
1. Use the tools for order-specific facts.
2. Never invent order information.
3. Use the return tool for return eligibility.
4. Use the issue tool for recorded product issues.
5. Keep answers concise and customer-friendly.
"""


def run_agent(question: str, max_iterations: int = 5) -> dict:
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=question),
    ]

    trace = []

    for _ in range(max_iterations):
        ai_message = llm_with_tools.invoke(messages)
        messages.append(ai_message)

        # No tool calls means we have the final answer.
        if not ai_message.tool_calls:
            return {
                "answer": ai_message.content,
                "messages": messages,
                "trace": trace,
            }

        for tool_call in ai_message.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            tool_call_id = tool_call["id"]

            if tool_name not in tool_map:
                tool_output = f"Unknown tool: {tool_name}"
            else:
                tool_output = tool_map[tool_name].invoke(tool_args)

            trace.append({
                "tool": tool_name,
                "arguments": tool_args,
                "output": str(tool_output),
            })

            messages.append(
                ToolMessage(
                    content=str(tool_output),
                    tool_call_id=tool_call_id,
                )
            )

    return {
        "answer": "Agent stopped because the maximum number of iterations was reached.",
        "messages": messages,
        "trace": trace,
    }

## Step 7 — Test one request

This question should cause more than one tool call because the customer asks about both:

- Return eligibility.
- Recorded issue.

In [ ]:
result = run_agent(
    "My order ORD1002 arrived damaged. Can I return it and what issue is recorded?"
)

print("FINAL ANSWER:")
print(result["answer"])

print("\nTOOL TRACE:")
display(pd.DataFrame(result["trace"]))

## Step 8 — Test different tool-selection scenarios

In [ ]:
questions = [
    "Where is ORD1001?",
    "How much did I pay for ORD1010?",
    "Is ORD1009 eligible for a return?",
    "Was any issue reported for ORD1013?",
]

for question in questions:
    result = run_agent(question)

    print("\nUSER:", question)
    print("AGENT:", result["answer"])
    print("TOOLS:", [x["tool"] for x in result["trace"]])

## Why this version fixes your error

Your earlier code imported:

```python
from langchain.tools import tool
from langchain.agents import create_agent
```

`create_agent` is built on LangGraph. Your environment currently has incompatible LangGraph/checkpoint/core components, which produces:

```text
TypeError: Reviver.__init__() got an unexpected keyword argument 'allowed_objects'
```

This notebook does not import `langchain.agents` or `langgraph`.

It only uses:

```text
langchain_openai
langchain_core
openai
pandas
python-dotenv
```

so it avoids the failing dependency path.